# AI Text Detection
This notebook loads data from dataset3_inputs.csv, makes predictions using GRU and ensemble models, and saves the results.

In [1]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import pickle
import re
import string
from nltk.tokenize import word_tokenize
import nltk
import absl.logging

absl.logging.set_verbosity(absl.logging.ERROR)

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

nltk.download("punkt", quiet=True)


2025-03-24 01:11:12.307315: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-24 01:11:12.323013: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742778672.339578  231382 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742778672.344200  231382 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-24 01:11:12.360662: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

True

## Define Helper Functions

In [2]:

def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    text = re.sub(r'[^\w\s.,!?]', '', text)
    tokens = word_tokenize(text)
    return " ".join(tokens)

def extract_features(text):
    if not isinstance(text, str):
        return [0, 0, 0, 0]
        
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    avg_sentence_length = np.mean([len(s.split()) for s in sentences]) if sentences else 0
    
    words = re.findall(r'\b\w+\b', text.lower())
    lexical_diversity = len(set(words)) / len(words) if words else 0
    
    punctuation_count = sum(1 for char in text if char in string.punctuation)
    punctuation_freq = punctuation_count / len(text) if text else 0
    
    first_person = len(re.findall(r'\b(I|me|my|mine|myself|we|us|our|ours|ourselves)\b', text, re.IGNORECASE))
    first_person_freq = first_person / len(words) if words else 0
    
    return [avg_sentence_length, lexical_diversity, punctuation_freq, first_person_freq]

def load_model(model_type="gru"):
    model_path = f"../trained_models/tensorflow/{model_type}_model.h5"
    
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file not found: {model_path}")
    
    model = tf.keras.models.load_model(model_path)
    
    with open("../preprocessed_tf/tokenizer.pkl", "rb") as f:
        tokenizer = pickle.load(f)
    
    with open("../preprocessed_tf/metadata.pkl", "rb") as f:
        metadata = pickle.load(f)
    
    return model, tokenizer, metadata

def predict_text(text, model_type="gru"):
    cleaned_text = clean_text(text)
    
    try:
        model, tokenizer, metadata = load_model(model_type)
    except FileNotFoundError as e:
        print(f"Error: {e}")
        print("Please train the model first by running train_tf.py")
        return None
    
    max_seq_length = metadata["max_seq_length"]
    
    sequence = tokenizer.texts_to_sequences([cleaned_text])
    padded_sequence = tf.keras.preprocessing.sequence.pad_sequences(
        sequence, maxlen=max_seq_length, padding='post', truncating='post'
    )
    
    if model_type == "ensemble":
        num_inputs = len(model.inputs)
        
        inputs = []
        for i in range(num_inputs):
            input_shape = model.inputs[i].shape
            
            if len(input_shape) == 2 and input_shape[1] == 4:
                features = extract_features(text)
                inputs.append(np.array([features]))
            else:
                inputs.append(padded_sequence)
        
        prediction = model.predict(inputs, verbose=0)[0][0]
    else:
        prediction = model.predict(padded_sequence, verbose=0)[0][0]
    
    predicted_class = "AI" if prediction >= 0.5 else "Human"
    
    return predicted_class


## Load Dataset

In [3]:

print("Loading dataset...")
try:
    df = pd.read_csv('../datasets/dataset3_inputs.csv', sep=';')
except:
    df = pd.read_csv('../datasets/dataset3_inputs.csv')

print(f"Dataset loaded with {len(df)} entries")

df.head()


Loading dataset...
Dataset loaded with 100 entries


,ID,Text
0,D3-1,String theory is a broad and varied subject th...
1,D3-2,String theory is a theoretical framework in ph...
2,D3-3,String theory proposes that the fundamental bu...
3,D3-4,I think string theory explains only the 3rd di...
4,D3-5,"With all this said, one should keep in mind th..."


## Make Predictions with GRU Model

In [4]:

print("Making predictions with GRU model...")
gru_predictions = []
for idx, row in df.iterrows():
    text = row['Text']
    prediction = predict_text(text, model_type="gru")
    gru_predictions.append(prediction)
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1}/{len(df)} entries with GRU model")

gru_results = pd.DataFrame({
    'ID': df['ID'],
    'Label': gru_predictions
})

gru_results.head()


Making predictions with GRU model...


2025-03-24 01:11:14.853316: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-03-24 01:11:14.853342: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:137] retrieving CUDA diagnostic information for host: jojocoelho-ASUS-TUF-Dash-F15-FX517ZE-FX517ZE
2025-03-24 01:11:14.853347: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:144] hostname: jojocoelho-ASUS-TUF-Dash-F15-FX517ZE-FX517ZE
2025-03-24 01:11:14.853487: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:168] libcuda reported version is: 550.120.0
2025-03-24 01:11:14.853502: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:172] kernel reported version is: 550.120.0
2025-03-24 01:11:14.853505: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:259] kernel version seems to match DSO: 550.120.0


Processed 10/100 entries with GRU model
Processed 20/100 entries with GRU model
Processed 30/100 entries with GRU model
Processed 40/100 entries with GRU model
Processed 50/100 entries with GRU model
Processed 60/100 entries with GRU model
Processed 70/100 entries with GRU model
Processed 80/100 entries with GRU model
Processed 90/100 entries with GRU model
Processed 100/100 entries with GRU model


,ID,Label
0,D3-1,Human
1,D3-2,AI
2,D3-3,AI
3,D3-4,Human
4,D3-5,Human


## Make Predictions with Ensemble Model

In [5]:

print("Making predictions with ensemble model...")
ensemble_predictions = []
for idx, row in df.iterrows():
    text = row['Text']
    prediction = predict_text(text, model_type="ensemble")
    ensemble_predictions.append(prediction)
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1}/{len(df)} entries with ensemble model")

ensemble_results = pd.DataFrame({
    'ID': df['ID'],
    'Label': ensemble_predictions
})

ensemble_results.head()


Making predictions with ensemble model...
Processed 10/100 entries with ensemble model
Processed 20/100 entries with ensemble model
Processed 30/100 entries with ensemble model
Processed 40/100 entries with ensemble model
Processed 50/100 entries with ensemble model
Processed 60/100 entries with ensemble model
Processed 70/100 entries with ensemble model
Processed 80/100 entries with ensemble model
Processed 90/100 entries with ensemble model
Processed 100/100 entries with ensemble model


,ID,Label
0,D3-1,Human
1,D3-2,AI
2,D3-3,AI
3,D3-4,Human
4,D3-5,Human


## Save Results

In [6]:

if not os.path.exists('results'):
    os.makedirs('results')

gru_results.to_csv('results/submissao2-grupo011-s1.csv', sep='\t', index=False)

ensemble_results.to_csv('results/submissao2-grupo011-s2.csv', sep='\t', index=False)

print("Predictions saved to results/submissao2-grupo011-s1.csv and results/submissao2-grupo011-s2.csv")


Predictions saved to results/submissao2-grupo011-s1.csv and results/submissao2-grupo011-s2.csv
